In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
np.random.seed(42)
X = np.linspace(0, 1, 100).reshape(-1, 1)
y = 2.5 * X.squeeze() + np.random.normal(0, 0.2, X.shape[0])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)
class BayesianLinearRegression:
  def __init__(self, alpha=1.0, beta=1.0):
    self.alpha = alpha
    self.beta = beta
    self.w_mean = None
    self.w_cov = None
  def fit(self, X, y):
    X = np.hstack((np.ones((X.shape[0], 1)), X))
    S0_inv = self.alpha * np.eye(X.shape[1])
    SN_inv = S0_inv + self.beta * X.T @ X
    SN = np.linalg.inv(SN_inv)
    mN = self.beta * SN @ X.T @ y
    self.w_mean = mN
    self.w_cov = SN
  def predict(self, X, return_std=False):
    X = np.hstack((np.ones((X.shape[0], 1)), X))
    mean = X @ self.w_mean
    if return_std:
      variance = 1 / self.beta + np.sum(X @ self.w_cov * X, axis=1)
      return mean, np.sqrt(variance)
    return mean
blr = BayesianLinearRegression(alpha=1.0, beta=25.0)
blr.fit(X_train, y_train)
X_pred = np.linspace(0, 1, 100).reshape(-1, 1)
y_pred, y_std = blr.predict(X_pred, return_std=True)
plt.figure(figsize=(10, 6))
plt.scatter(X, y, label="Data", color="blue", alpha=0.6)
plt.plot(X_pred, y_pred, label="Prediction (mean)", color="red")
plt.fill_between(
    X_pred.squeeze(),
    y_pred - 2 * y_std,
    y_pred + 2 * y_std,
    color="pink",
    alpha=0.4,
    label="Uncertainty (95%)", )
plt.title("Bayesian Linear Regression")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()